In [1]:
from dotenv import load_dotenv
import os
api_key = os.getenv("ROBOFLOW_API_KEY")

In [2]:
BATCH_SIZE = 16

In [3]:
# !pip install roboflow

In [4]:
# !pip install typing_extensions==4.15.0 torch==2.7.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

In [5]:
from roboflow import Roboflow
rf = Roboflow(api_key=api_key)
project = rf.workspace("crater-zqpjg").project("crater-vrqmn")
version = project.version(1)
dataset = version.download("coco")

loading Roboflow workspace...
loading Roboflow project...


In [6]:
dataset_path = "./crater-1"

In [7]:
import os

for split in ["train", "valid", "test"]:
    images = os.listdir(f"{dataset_path}/{split}")
    print(f"{split}: {len(images)} images")

train: 2490 images
valid: 712 images
test: 357 images


In [8]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import json

In [9]:
class COCODataset(Dataset):
    def __init__(self, images_dir, annotations_file, transforms=None):
        self.images_dir = images_dir
        self.transforms = transforms

        # Load COCO annotations JSON
        with open(annotations_file, "r") as f:
            coco = json.load(f)

        # Map image_id -> image info
        self.images = {img["id"]: img for img in coco["images"]}
        self.image_ids = list(self.images.keys())

        # Map image_id -> list of annotations
        self.annotations = {}
        for ann in coco["annotations"]:
            img_id = ann["image_id"]
            self.annotations.setdefault(img_id, []).append(ann)

        self.categories = {cat["id"]: cat["name"] for cat in coco["categories"]}

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_info = self.images[image_id]

        # Load image
        img_path = os.path.join(self.images_dir, image_info["file_name"])
        image = Image.open(img_path).convert("RGB")

        # Get annotations for this image
        anns = self.annotations.get(image_id, [])

        boxes, labels, areas, iscrowd = [], [], [], []

        for ann in anns:
            x, y, w, h = ann["bbox"]
            boxes.append([x, y, x + w, y + h])   # convert to [x1, y1, x2, y2]
            labels.append(ann["category_id"])
            areas.append(ann["area"])
            iscrowd.append(ann.get("iscrowd", 0))

        target = {
            "image_id": torch.tensor([image_id]),
            "boxes":    torch.tensor(boxes,   dtype=torch.float32) if boxes else torch.zeros((0, 4)),
            "labels":   torch.tensor(labels,  dtype=torch.int64),
            "area":     torch.tensor(areas,   dtype=torch.float32),
            "iscrowd":  torch.tensor(iscrowd, dtype=torch.int64),
        }

        if self.transforms:
            image = self.transforms(image)

        return image, target

In [10]:
import torchvision.transforms as T

# Transforms — train gets augmentation, valid/test just get normalized
train_transforms = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

eval_transforms = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])


train_dataset = COCODataset(
    images_dir=f"{dataset_path}/train",
    annotations_file=f"{dataset_path}/train/_annotations.coco.json",
    transforms=train_transforms
)

valid_dataset = COCODataset(
    images_dir=f"{dataset_path}/valid",
    annotations_file=f"{dataset_path}/valid/_annotations.coco.json",
    transforms=eval_transforms
)

test_dataset = COCODataset(
    images_dir=f"{dataset_path}/test",
    annotations_file=f"{dataset_path}/test/_annotations.coco.json",
    transforms=eval_transforms
)

In [11]:
def collate_fn(batch):
    # Needed because each image can have a different number of boxes
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [12]:
images, targets = next(iter(train_loader))

print(f"Batch size     : {len(images)}")
print(f"Image shape    : {images[0].shape}")         # [3, H, W]
print(f"Boxes (img 0)  : {targets[0]['boxes']}")
print(f"Labels (img 0) : {targets[0]['labels']}")

Batch size     : 16
Image shape    : torch.Size([3, 416, 416])
Boxes (img 0)  : tensor([[173., 259., 271., 357.],
        [203., 106., 234., 136.],
        [327.,  44., 342.,  58.]])
Labels (img 0) : tensor([1, 1, 1])


In [13]:
import json

def count_object_sizes(annotations_file):
    with open(annotations_file) as f:
        coco = json.load(f)

    small, medium, large = 0, 0, 0

    for ann in coco["annotations"]:
        w, h = ann["bbox"][2], ann["bbox"][3]
        area = w * h

        if area < 32**2:
            small += 1
        elif area < 96**2:
            medium += 1
        else:
            large += 1

    total = small + medium + large

    print(f"Total objects : {total}")
    print(f"Small         : {small}  ({100*small/total:.1f}%)")
    print(f"Medium        : {medium} ({100*medium/total:.1f}%)")
    print(f"Large         : {large}  ({100*large/total:.1f}%)")

    return {"small": small, "medium": medium, "large": large, "total": total}

In [14]:
for split in ["train", "valid", "test"]:
    print(f"\n── {split.upper()} ──")
    count_object_sizes(f"{dataset_path}/{split}/_annotations.coco.json")


── TRAIN ──
Total objects : 4953
Small         : 3243  (65.5%)
Medium        : 1378 (27.8%)
Large         : 332  (6.7%)

── VALID ──
Total objects : 1364
Small         : 862  (63.2%)
Medium        : 431 (31.6%)
Large         : 71  (5.2%)

── TEST ──
Total objects : 731
Small         : 467  (63.9%)
Medium        : 219 (30.0%)
Large         : 45  (6.2%)


In [15]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def get_model(num_classes):
    # num_classes = 1 class + 1 background = 2
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")  # pretrained backbone

    # Replace the classifier head for our number of classes
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    return model

model = get_model(num_classes=2)  # background + your 1 class
model.to(device)

Using device: cuda
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to C:\Users\DIAT/.cache\torch\hub\checkpoints\fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:01<00:00, 102MB/s] 


FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

In [16]:
next(model.parameters()).device

device(type='cuda', index=0)

In [17]:
params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

# Decays LR by 0.1 every 3 epochs
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)

In [18]:
def train_one_epoch(model, optimizer, dataloader, device, epoch):
    model.train()
    total_loss = 0

    for batch_idx, (images, targets) in enumerate(dataloader):
        # Move to device
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward pass — model returns loss dict in train mode
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        # Backward pass
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()

        if batch_idx % 10 == 0:
            print(f"Epoch [{epoch}] Batch [{batch_idx}/{len(dataloader)}] "
                  f"Loss: {losses.item():.4f} "
                  f"(cls: {loss_dict['loss_classifier'].item():.4f}, "
                  f"box: {loss_dict['loss_box_reg'].item():.4f}, "
                  f"rpn_cls: {loss_dict['loss_objectness'].item():.4f}, "
                  f"rpn_box: {loss_dict['loss_rpn_box_reg'].item():.4f})")

    return total_loss / len(dataloader)

In [19]:
@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0

    for images, targets in dataloader:
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Trick: model only returns losses in train mode
        # so we temporarily switch for loss computation
        model.train()
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        total_loss += losses.item()
        model.eval()

    return total_loss / len(dataloader)

In [20]:
NUM_EPOCHS = 20
best_val_loss = float("inf")

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)
    val_loss   = evaluate(model, valid_loader, device)
    lr_scheduler.step()

    print(f"\nEpoch [{epoch}/{NUM_EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f}\n")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_fasterrcnn.pth")
        print(f"  ✅ Saved best model (val_loss: {val_loss:.4f})")

print("Training complete!")

Epoch [1] Batch [0/156] Loss: 1.6013 (cls: 1.2416, box: 0.0936, rpn_cls: 0.2474, rpn_box: 0.0187)
Epoch [1] Batch [10/156] Loss: 0.2367 (cls: 0.0628, box: 0.0484, rpn_cls: 0.1160, rpn_box: 0.0095)
Epoch [1] Batch [20/156] Loss: 0.3006 (cls: 0.1082, box: 0.0748, rpn_cls: 0.1078, rpn_box: 0.0097)
Epoch [1] Batch [30/156] Loss: 0.2450 (cls: 0.0650, box: 0.0684, rpn_cls: 0.1033, rpn_box: 0.0083)
Epoch [1] Batch [40/156] Loss: 0.2174 (cls: 0.0755, box: 0.0734, rpn_cls: 0.0636, rpn_box: 0.0049)
Epoch [1] Batch [50/156] Loss: 0.2091 (cls: 0.0743, box: 0.0585, rpn_cls: 0.0674, rpn_box: 0.0090)
Epoch [1] Batch [60/156] Loss: 0.2688 (cls: 0.0785, box: 0.1216, rpn_cls: 0.0588, rpn_box: 0.0098)
Epoch [1] Batch [70/156] Loss: 0.1605 (cls: 0.0654, box: 0.0435, rpn_cls: 0.0479, rpn_box: 0.0038)
Epoch [1] Batch [80/156] Loss: 0.1842 (cls: 0.0603, box: 0.0569, rpn_cls: 0.0629, rpn_box: 0.0041)
Epoch [1] Batch [90/156] Loss: 0.1849 (cls: 0.0635, box: 0.0773, rpn_cls: 0.0391, rpn_box: 0.0050)
Epoch [1] B

In [21]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

@torch.no_grad()
def evaluate_map(model, dataloader, device):
    model.eval()

    metric = MeanAveragePrecision(
        iou_type="bbox",
        iou_thresholds=[0.5, 0.75],
        max_detection_thresholds=[1, 10, 100]
    )

    for images, targets in dataloader:
        images = [img.to(device) for img in images]
        predictions = model(images)

        preds = [{
            "boxes":  pred["boxes"].cpu(),
            "scores": pred["scores"].cpu(),
            "labels": pred["labels"].cpu(),
        } for pred in predictions]

        gts = [{
            "boxes":  t["boxes"].cpu(),
            "labels": t["labels"].cpu(),
        } for t in targets]

        metric.update(preds, gts)

    return metric.compute()

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [22]:
model.load_state_dict(torch.load("best_fasterrcnn.pth", map_location=device))
results = evaluate_map(model, test_loader, device)

# --- 1. Compute F1-Score ---
# Get the overall mAP and the mAR at max 100 detections
map_val = results.get("map", torch.tensor(0.0)).item()
mar_val = results.get("mar_100", torch.tensor(0.0)).item()

# Calculate harmonic mean for F1-score (add epsilon to avoid division by zero)
if (map_val + mar_val) > 0:
    f1_score = 2 * (map_val * mar_val) / (map_val + mar_val + 1e-8)
else:
    f1_score = 0.0

# --- 2. Define all metrics to print ---
metrics_to_print = {
    # Precision Metrics
    "mAP@0.50:0.95 (all)"    : "map",
    "mAP@0.50      (all)"    : "map_50",
    "mAP@0.75      (all)"    : "map_75",
    "mAP@0.50:0.95 (small)"  : "map_small",
    "mAP@0.50:0.95 (medium)" : "map_medium",
    "mAP@0.50:0.95 (large)"  : "map_large",
    
    # Recall Metrics (corresponds to your max_detection_thresholds)
    "mAR@0.50:0.95 (max=1)"  : "mar_1",
    "mAR@0.50:0.95 (max=10)" : "mar_10",
    "mAR@0.50:0.95 (max=100)": "mar_100",
    "mAR@0.50:0.95 (small)"  : "mar_small",
    "mAR@0.50:0.95 (medium)" : "mar_medium",
    "mAR@0.50:0.95 (large)"  : "mar_large",
}

# --- 3. Print the table ---
print("=" * 45)
print(f"{'Metric':<30} {'Value':>10}")
print("=" * 45)

for label, key in metrics_to_print.items():
    val = results.get(key, torch.tensor(float("nan"))).item()
    print(f"{label:<30} {val:>10.4f}")

print("-" * 45)
# Print the custom F1-score
print(f"{'F1-Score (mAP & mAR@100)':<30} {f1_score:>10.4f}")
print("=" * 45)

Metric                              Value
mAP@0.50:0.95 (all)                0.6134
mAP@0.50      (all)                0.8684
mAP@0.75      (all)                0.3583
mAP@0.50:0.95 (small)              0.4677
mAP@0.50:0.95 (medium)             0.7928
mAP@0.50:0.95 (large)              0.9178
mAR@0.50:0.95 (max=1)              0.3735
mAR@0.50:0.95 (max=10)             0.7045
mAR@0.50:0.95 (max=100)            0.7196
mAR@0.50:0.95 (small)              0.6439
mAR@0.50:0.95 (medium)             0.8455
mAR@0.50:0.95 (large)              0.9444
---------------------------------------------
F1-Score (mAP & mAR@100)           0.6622
